In [ ]:
from sqlalchemy import create_engine, Column, Integer, String,Float,DateTime,MetaData,select,BigInteger,ForeignKey
from sqlalchemy.orm import declarative_base,relationship,sessionmaker

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

In [ ]:
print(db_user)

In [ ]:
database_url = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(database_url,connect_args={"options": "-csearch_path=endeavour"})

In [ ]:
Base= declarative_base()

In [ ]:
#company locations staging
class CompanyLocationsStaging(Base):
    __tablename__='company_locations_staging'
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol= Column(String(10),primary_key=True)
    address=Column(String(50))
    city=Column(String(100))
    state=Column(String(50))
    zip=Column(String(50))
    country=Column(String(100))

In [ ]:
#sector_lookup
class SectorLookup(Base):
    __tablename__ = "sector_lookup"
    __table_args__ = {"schema": "endeavour"}
    sector_id = Column(Integer, primary_key=True)
    sector_name = Column(String(25))
    stock_fundamentals = relationship(
        "StockFundamentals",
        back_populates="sector"
    )
    subsectors = relationship(
        "SubsectorLookup",
        back_populates="sector"
    )

In [ ]:
#state lookup
class StateLookup(Base):
    __tablename__='state_lookup'
    __table_args__ = {"schema": "endeavour"}
    state_symbol= Column(String(10),primary_key=True)
    state_name=Column(String(25))

In [ ]:
#stock_fundamentals_staging
class StockFundamentalsStaging(Base):
    __tablename__='stock_fundamentals_staging'
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol= Column(String(10),primary_key=True)
    sector_id=Column(Integer)
    subsector_id=Column(Integer)
    market_cap=Column(BigInteger)
    current_ratio=Column(Float)
    price_to_book_ratio=Column(Float)
    peg=Column(Float)
    epsqqq=Column(Float)
    eps_nxtyear=Column(Float)
    eps_ttm=Column(Float)
    roe=Column(Float)
    insider_ownership=Column(Float)
    debt_equity_ratio=Column(Float)
    trailing_pe=Column(Float)
    forward_pe=Column(Float)

In [ ]:
#stocks Pricing history staging
class StocksPriceHistoryStaging(Base):
    __tablename__='stocks_price_history_staging'
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol= Column(String(10),primary_key=True)
    trading_date=Column(DateTime,primary_key=True)
    open_price=Column(Float)
    high_price=Column(Float)
    low_price=Column(Float)
    volume=Column(BigInteger)
    dividends=Column(BigInteger)
    close_price=Column(Float)

In [ ]:
#total_market_stocks
class TotalMarketStocks(Base):
    __tablename__ = "total_market_stocks"
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol = Column(String(10), primary_key=True)
    ticker_name = Column(String(75))

In [ ]:
#stocks lookup
class StocksLookup(Base):
    __tablename__ = "stocks_lookup"
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol = Column(String(10), primary_key=True)
    ticker_name = Column(String(75))
    stock_fundamentals = relationship(
        "StockFundamentals",
        back_populates="stock_lookup"
    )
    stocks_price_history = relationship(
        "StocksPriceHistory",
        back_populates="stock"
    )
    company_location = relationship(
        "CompanyLocations",
        back_populates="stock"
    )

In [ ]:
#stock_fundamentals


class StockFundamentals(Base):
    __tablename__ = "stock_fundamentals"
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol = Column(
        String(10),
        ForeignKey("endeavour.stocks_lookup.ticker_symbol"),
        primary_key=True
    )

    sector_id = Column(Integer, ForeignKey("endeavour.sector_lookup.sector_id"))
    subsector_id = Column(Integer, ForeignKey("endeavour.subsector_lookup.subsector_id"))

    market_cap = Column(BigInteger)
    current_ratio = Column(Float)
    price_to_book_ratio = Column(Float)
    peg = Column(Float)
    epsqqq = Column(Float)
    eps_nxtyear = Column(Float)
    eps_ttm = Column(Float)
    roe = Column(Float)
    insider_ownership = Column(Float)
    debt_equity_ratio = Column(Float)
    trailing_pe = Column(Float)
    forward_pe = Column(Float)

    # ✅ FIXED RELATIONSHIPS


    stock_lookup = relationship(
        "StocksLookup",
        back_populates="stock_fundamentals"
    )

    sector = relationship(
        "SectorLookup",
        back_populates="stock_fundamentals"
    )

    subsector = relationship(
        "SubsectorLookup",
        back_populates="stock_fundamentals"
    )

In [ ]:
#stocks_price_history
class StocksPriceHistory(Base):
    __tablename__ = "stocks_price_history"
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol = Column(
        String(10),
        ForeignKey("endeavour.stocks_lookup.ticker_symbol"),
        primary_key=True
    )

    trading_date = Column(DateTime, primary_key=True)

    open_price = Column(Float)
    high_price = Column(Float)
    low_price = Column(Float)
    volume = Column(BigInteger)
    dividends = Column(BigInteger)
    close_price = Column(Float)

    stock = relationship(
        "StocksLookup",
        back_populates="stocks_price_history"
    )

In [ ]:
#subsector_lookup
class SubsectorLookup(Base):
    __tablename__ = "subsector_lookup"
    __table_args__ = {"schema": "endeavour"}
    subsector_id = Column(Integer, primary_key=True)
    subsector_name = Column(String(75))

    sector_id = Column(
        Integer,
        ForeignKey("endeavour.sector_lookup.sector_id")
    )

    sector = relationship(
        "SectorLookup",
        back_populates="subsectors"
    )
    stock_fundamentals = relationship(
        "StockFundamentals",
        back_populates="subsector"
    )

In [ ]:
#company_locations
class CompanyLocations(Base):
    __tablename__ = "company_locations"
    __table_args__ = {"schema": "endeavour"}
    ticker_symbol = Column(
        String(10),
        ForeignKey("endeavour.stocks_lookup.ticker_symbol"),
        primary_key=True
    )

    address = Column(String(50))
    city = Column(String(100))
    state = Column(String(50))
    zip = Column(String(50))
    country = Column(String(100))

    stock=relationship(
        "StocksLookup",
        back_populates="company_location"
    )

In [ ]:
SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)
inspector.get_table_names(schema='endeavour')

In [ ]:
inspector.get_table_names()

In [ ]:
inspector.get_table_names(schema="endeavour")

In [ ]:
print(engine.url.database)

In [ ]:
s = session.query(CompanyLocations).first()
print(s)

In [ ]:
stocks = session.query(StocksLookup).limit(10).all()

for s in stocks:
    print({c.name: getattr(s, c.name) for c in s.__table__.columns})